# TrifoolPercent Dev Log (notebook edition)

This notebook mirrors [`devlog.md`](devlog.md) and adds things a plain Markdown file can't carry: inline screenshots, small runnable snippets, and (eventually) live renders of exported scenes and textures. GitHub renders `.ipynb` natively, so it reads in a browser without a kernel.

**Which file is canonical?** `devlog.md` is the single source of truth for the running narrative, because it is friction-free to edit from any shell, merges cleanly, and reviews well. This notebook is a *companion*: it re-presents the same entries with embedded media and runnable inspection cells. If the two drift, trust `devlog.md`.

**Tradeoffs noted up front:**

- ✅ GitHub renders notebooks in-browser; no kernel needed to read.
- ✅ Cells can embed screenshots, outputs of Torch/Fast64 steps, scene stats, texture previews.
- ⚠️ Diffs are noisy: cell outputs and metadata churn even when prose doesn't change. Strip outputs with `jupyter nbconvert --clear-output --inplace docs/devlog.ipynb` before committing.
- ⚠️ Merge conflicts in `.ipynb` are painful. Keep the narrative in `devlog.md`; use this notebook for *showing* things, not for the primary edit stream.
- ⚠️ "Live renders" (Blender/Fast64, SoH screenshots from a running build) still need a machine with the toolchain. The notebook can display their outputs, but it isn't a replacement for the real build pipeline — it's a thin viewer on top of it.

See [`timeline.md`](timeline.md) for the phase milestone view and [`porting-checklist.md`](porting-checklist.md) for the mechanic-level checklist.

## 2026-10-15 — Catching up: where the project stands today

Phase 1 (Triforce Room PoC) is live in the `besitu` debug scene slot. The scene, collision, water, indoor lighting, title card and exit are confirmed in a stock Ship of Harkinian 9.2.3 Windows build with the `.o2r` alone. The SoH module (three actors + cutscene + text) compiles clean against Shipwright `6b4edce` but hasn't been exercised in a running game yet because no Windows SoH build with the module linked in has been produced.

The sections below mirror the current entry in [`devlog.md`](devlog.md); read that for the full narrative. Below the Markdown, two inspection cells show how to peek at the current `.o2r` and screenshots without launching SoH.

### Mistakes already paid for (short list)

- Don't hijack the Goron Shop slot — use a debug slot (`besitu`).
- Triforce%'s collision is floors+stairs only; rebuild from room geometry.
- Five scene textures were never published upstream; placeholders are expected.
- Anything that mutates state belongs in `Update`, not `Draw` (frame interpolation).
- Don't overwrite actor slots — register with `ActorDB` and spawn from `OnSceneSpawnActors`.
- Objects: use Triforce%'s compiled `zobj.zobj` through Torch, not a Fast64 re-export.
- Use HarbourMasters' Fast64 fork, not stock Fast64.
- Fetch Shipwright submodules by SHA in CI, not via `actions/checkout`.
- Scene-slot behaviour follows the *slot*, not the mod (item blocking, music fallback, etc.).

See [`devlog.md`](devlog.md) for the reasons.

### Peek inside the current `.o2r`

`.o2r` files are zip archives of SoH resources. This cell lists what the current build (or the committed PoC archive) contains, so you can see at a glance which scene files, objects and textures are packed. Runs with the Python standard library only.

In [ ]:
import os, zipfile, pathlib

repo = pathlib.Path('..').resolve()
candidates = [
    repo / 'build' / 'TrifoolPercent.o2r',
    repo / 'triforce-room-poc' / 'TrifoolPercent.o2r',
]
archive = next((p for p in candidates if p.exists()), None)

if archive is None:
    print('No .o2r found. Run tools/build_o2r.sh to produce build/TrifoolPercent.o2r,')
    print('or check triforce-room-poc/ for a committed snapshot.')
else:
    print(f'Reading {archive.relative_to(repo)}')
    with zipfile.ZipFile(archive) as zf:
        names = sorted(zf.namelist())
        print(f'  {len(names)} resources')
        for n in names:
            print('   ', n)

### Reference screenshots

Reference images from the original Triforce% run live in [`screenshots/`](../screenshots/). The next cell lists them; drop the path into a Markdown image tag (e.g. `![](../screenshots/<name>)`) to embed one inline here.

In [ ]:
import pathlib

shots = pathlib.Path('..') / 'screenshots'
if not shots.exists():
    print('No screenshots/ directory yet.')
else:
    for p in sorted(shots.rglob('*')):
        if p.is_file():
            print(p.relative_to(shots.parent))

## How to add an entry

1. Add the new section to [`devlog.md`](devlog.md) first (that's the canonical copy).
2. Mirror it here with any screenshots, cell outputs or renders worth preserving.
3. Before committing, clear outputs:
   ```sh
   jupyter nbconvert --clear-output --inplace docs/devlog.ipynb
   ```
   This keeps diffs sane. If a cell's output *is* the point of the entry (a rendered texture, a chart), save it as a file under `docs/` or `screenshots/` and embed it with Markdown instead of relying on cell output.